# Ours (final All-CDR) — Zero-shot & Few-shot Transfer
## Six external single-antigen datasets: 1mlc, 1n8z, 4fqi, koenig, trastuzumab, warszawski

All-CDR SAaIntDB weights. Zero-shot = 10-fold All-CDR ensemble; few-shot = fine-tune at **10/20/30%** with **3 seeds** (mean ± std). 4fqi subsampled to 4000 variants.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
HERE=os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(HERE,'requirements.txt')) and os.path.dirname(HERE)!=HERE:
    HERE=os.path.dirname(HERE)  # walk up to the repo root (launch Jupyter anywhere)
plt.rcParams.update({'font.family':'Arial','font.size':7,'axes.titlesize':8,'axes.labelsize':7,
  'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,'legend.frameon':False,
  'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':400,'savefig.dpi':400,
  'savefig.bbox':'tight','pdf.fonttype':42})
FIG=os.path.join(HERE,'figures_transfer'); os.makedirs(FIG,exist_ok=True)
def save_fig(f,n):
    for e in ('pdf','png'): f.savefig(os.path.join(FIG,f'{n}.{e}'),bbox_inches='tight')
    print('saved',n)
agg=pd.read_csv(os.path.join(HERE,'experiments','results_transfer_allcdr','aggregated.csv'))
DS=['1mlc','1n8z','4fqi','koenig','trastuzumab','warszawski']
print(agg.to_string(index=False))

---
## 1. Few-shot Learning Curves (per dataset, 3 seeds)

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(10,6)); axes=axes.flatten()
for ax,ds,letter in zip(axes,DS,'ABCDEF'):
    a=agg[agg.dataset==ds].sort_values('fraction')
    fs=a[a.fraction>0]
    zs=a[a.fraction==0]['pearson_mean'].values[0]
    ax.axhline(zs,color='#95a5a6',ls=':',lw=1,label=f'zero-shot ({zs:.2f})')
    ax.errorbar(fs.fraction*100,fs.pearson_mean,yerr=fs.pearson_std,fmt='o-',color='#e74c3c',lw=1.8,ms=5,capsize=3,label='few-shot')
    ax.set_xlabel('Training data (%)'); ax.set_ylabel('Pearson r' if letter in 'AD' else '')
    ax.set_title(f'{letter}  {ds}',fontsize=8); ax.legend(fontsize=6); ax.set_xticks([10,20,30])
    ax.text(0.02,0.97,letter,transform=ax.transAxes,fontsize=10,fontweight='bold',va='top')
fig.suptitle('All-CDR transfer: few-shot learning curves (mean ± std, 3 seeds)',fontsize=9,fontweight='bold')
plt.tight_layout(); save_fig(fig,'transfig1_fewshot_curves'); plt.show()

---
## 2. Zero-shot vs Few-shot Summary (all metrics)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(10,3.2))
x=np.arange(len(DS)); w=0.2
fracs=[0.0,0.1,0.2,0.3]; cols=['#95a5a6','#f39c12','#3498db','#e74c3c']; labs=['zero','10%','20%','30%']
for ax,metric,ylab,letter in [(axes[0],'pearson_mean','Pearson r','A'),(axes[1],'spearman_mean','Spearman ρ','B'),(axes[2],'rmse_mean','RMSE','C')]:
    for j,(fr,c,lb) in enumerate(zip(fracs,cols,labs)):
        vals=[float(agg[(agg.dataset==d)&(agg.fraction==fr)][metric].values[0]) if len(agg[(agg.dataset==d)&(agg.fraction==fr)]) else np.nan for d in DS]
        ax.bar(x+(j-1.5)*w,vals,w,color=c,alpha=0.85,label=lb if letter=='A' else None)
    ax.set_xticks(x); ax.set_xticklabels(DS,rotation=30,ha='right',fontsize=6); ax.set_ylabel(ylab)
    ax.set_title(f'{letter}  {ylab}',fontsize=8)
    if metric=='rmse_mean': ax.set_yscale('log')
    if letter=='A': ax.legend(fontsize=6,title='train frac')
    ax.text(0.02,0.97,letter,transform=ax.transAxes,fontsize=10,fontweight='bold',va='top')
fig.suptitle('All-CDR transfer across 6 datasets — zero-shot vs few-shot',fontsize=9,fontweight='bold')
plt.tight_layout(); save_fig(fig,'transfig2_summary_allmetrics'); plt.show()

---
## 3. Summary Table

In [ ]:
print(agg.round(4).to_string(index=False))